<img src='../OUTILS/bandeau_MF.png' align='right' width='100%'/>

# 🌍🛰️ Téléchargement et visualisation de données MTG avec EUMDAC

---

## 🎯 Objectifs du TP

- 🔑 Utiliser l'EUMETSAT Data Access Client (EUMDAC) pour télécharger des données MTG
- 🔎 Explorer la structure des fichiers NetCDF avec GDAL
- 🖼️ Produire des images à partir de canaux spécifiques (ex. VIS006)
- 🧩 Assembler plusieurs chunks pour reconstituer une zone géographique
- ✂️ Découper et personnaliser des produits directement via le service Data Tailor

---

## 📑 Table des matières

| Section | Titre |
|---------|-------|
| 1 | [Configuration et prérequis](#1-configuration-et-prerequis) |
| 2 | [Recherche et téléchargement de données](#2-recherche-et-telechargement-de-donnees) |
| 3 | [Lecture des métadonnées avec GDAL](#3-lecture-des-metadonnees-avec-gdal) |
| 4 | [Extraction d’un canal et création d’une image](#4-extraction-dun-canal-et-creation-dune-image) |
| 5 | [Téléchargement et assemblage de plusieurs chunks](#5-telechargement-et-assemblage-de-plusieurs-chunks) |
| 6 | [Visualisation d’un plein disque](#6-visualisation-dun-plein-disque) |
| 7 | [Téléchargement direct d’une sous-zone (Data Tailor)](#7-telechargement-direct-dune-sous-zone-data-tailor) |

---

<div class="alert alert-block alert-warning">
    
### ⚠️ PRÉREQUIS
    
- Être enregistré auprès d’EUMETSAT et disposer d’un **login** et d’un **mot de passe** (User credentials).   </br>
  → Obtenez-les sur [https://api.eumetsat.int/api-key/#](https://api.eumetsat.int/api-key/#) </br>
  Ensuite, vérifier dans My profile / My data licenses que les différentes licences sont actives </br>
  <a href="../DOCS/eumetsat_user_portal_data_licenses_0.jpg" target="_blank"> <img src='../DOCS/eumetsat_user_portal_data_licenses_0.jpg' align='left' width='20%'/> </a> </br></br></br></br></br>
  <a href="../DOCS/eumetsat_user_portal_data_licenses.jpg" target="_blank"> <img src='../DOCS/eumetsat_user_portal_data_licenses.jpg' align='left' width='20%'/> </a> </br></br></br></br></br></br>
- Le kernel `env_MF_teledetection` doit être actif (contenant `eumdac`, `gdal`, `PIL`, etc.).
</div>
<hr>

<div class="alert alert-info" role="alert">
    
### 📖 Utilisation de l’EUMETSAT Data Access Client (EUMDAC)

EUMDAC est le client officiel d’accès aux données d’EUMETSAT. Il permet de rechercher, télécharger et personnaliser des produits (Data Store, Data Tailor).  
**Documentation utile :**
- [Guide EUMDAC](https://user.eumetsat.int/resources/user-guides/eumetsat-data-access-client-eumdac-guide)
- [Data Store – présentation](https://user.eumetsat.int/data-access/data-store)
- [Catalogue des données](https://data.eumetsat.int/search)

</div>

<div class="alert alert-info alert-success">
    
## 1 - Configuration et prérequis
</div>

### 📦 Importation des librairies et paramétrage de l’environnement

In [ ]:
from datetime import datetime
import sys
import os
import glob
import subprocess
from osgeo import gdal
from PIL import Image
from IPython.display import display

# Configuration des chemins pour GDAL/PROJ dans l'environnement conda
os.environ['PATH'] = f"/opt/conda/env_MF_teledetection/bin:{os.environ['PATH']}" 
os.environ['PATH'] = f"~/.conda/envs/env_MF_teledetection/bin:{os.environ['PATH']}"
os.environ['GDAL_DATA'] = '/opt/conda/env_MF_teledetection/share/gdal'
os.environ['PROJ_LIB'] = '/opt/conda/env_MF_teledetection/share/proj'

print("✅ Environnement prêt.")

### 📂 Se positionner dans le répertoire de travail `MF_DATA_MANIPULATION`

In [ ]:
work_dir = os.path.expanduser("~/MF_DATA_MANIPULATION")
os.makedirs(work_dir, exist_ok=True)
os.chdir(work_dir)
print(f"📁 Répertoire courant : {os.getcwd()}")

### 🔧 Installation / mise à jour d’EUMDAC (si nécessaire)

Le client EUMDAC peut être installé via `conda install -c eumetsat eumdac`.  
Nous allons également cloner le dépôt Git pour disposer des exemples (optionnel).

In [ ]:
if not os.path.isdir("eumdac"):
    !git clone https://gitlab.eumetsat.int/eumetlab/data-services/eumdac.git
    print("📥 Dépôt eumdac cloné.")
else:
    print("📁 Dépôt eumdac déjà présent.")

### 🔐 Configuration des identifiants EUMETSAT

 **2 possibilités :**

- Lancer cette commande : eumdac set-credentials <em>votre_login</em> <em>votre_mot_de_passe</em> > **Nous choisissons cette solution pour ce TP** </br>
- Créer un fichier .netrc sous votre home, en utilsant vos informations login et password que vous avez récupéré, contenant ces 3 lignes :</br>
machine api.eumetsat.int</br>
login <em>votre_login</em></br>
password <em>votre_mot_de_passe</em></br>

In [ ]:
!eumdac set-credentials votre_login votre_mot_de_passe

<div class="alert alert-info alert-success">
    
## 2 - Recherche et téléchargement de données
</div>

### 🛰️ Collections MTG disponibles

| Collection | Description | Identifiant |
|------------|-------------|-------------|
| FCI L1c Normal Resolution | Résolution nominale (2 km) | `EO:EUM:DAT:0662` |
| FCI L1c High Resolution   | Résolution 1 km (canaux visibles) | `EO:EUM:DAT:0665` |
| FCI L1c RGB (GeoColour)   | Produit L1.5 | `EO:EUM:DAT:0913` |

Quelques liens utiles :  </br>
Catalogue des données Eumetsat : https://data.eumetsat.int/extended?query=mtg  </br>
FCI Level 1c Normal Resolution Image Data - MTG - 0 degree : https://data.eumetsat.int/data/map/EO:EUM:DAT:0662  </br>
FCI Level 1c High Resolution Image Data - MTG - 0 degree   : https://data.eumetsat.int/data/map/EO:EUM:DAT:0665

**Catalogue interactif** : [https://data.eumetsat.int/extended?query=mtg](https://data.eumetsat.int/extended?query=mtg)

### 💻 Commandes utiles d’EUMDAC

| Commande | Description |
|----------|-------------|
| `eumdac search -c COLLECTION --limit 5` | Affiche les 5 derniers produits |
| `eumdac download -c COLLECTION --limit 1` | Télécharge le dernier produit complet |
| `eumdac download -c COLLECTION --entry '*0036.nc'` | Télécharge seulement un chunk spécifique |
| `eumdac download -c COLLECTION --start YYYY-MM-DDTHH:MM:SS --end ...` | Filtrage temporel |
| `eumdac describe` | Affiche toutes les options |

### 🔍 Exemple : visualiser les 5 derniers fichiers de la collection Normal Resolution

In [ ]:
!eumdac search -c EO:EUM:DAT:0662 --limit 5

### ⬇️ Télécharger un seul chunk (exemple : chunk n°36, date 2026-04-22, 11h00 UTC)

In [ ]:
output_chunk_dir = os.path.join(work_dir, "CHUNKS")
os.makedirs(output_chunk_dir, exist_ok=True)

!eumdac download -c EO:EUM:DAT:0662 \
    --start 2026-04-22T11:00:00 --end 2026-04-22T11:10:00 \
    --entry '*0036.nc' \
    --output {output_chunk_dir}

<div class="alert alert-info alert-success">
    
## 3 - Lecture des métadonnées avec GDAL
</div>

### 🗂️ Localiser le fichier téléchargé

Les noms de dossiers/fichiers étant très longs, nous allons les stocker dans des variables Python.

In [ ]:
# Trouver le répertoire contenant les chunks (un seul sous-dossier attendu)
chunk_folders = glob.glob(os.path.join(output_chunk_dir, "W_XX-EUMETSAT-Darmstadt*"))
if chunk_folders:
    repertoire = chunk_folders[0]
else:
    # Répertoire de secours (données pré-téléchargées)
    repertoire = "/stockage/DATA/202505141000"
    print("⚠️ Utilisation du répertoire de secours :", repertoire)

# Trouver un fichier .nc dans ce répertoire
nc_files = glob.glob(os.path.join(repertoire, "*.nc"))
if nc_files:
    fichier = os.path.basename(nc_files[0])
else:
    raise FileNotFoundError("Aucun fichier NetCDF trouvé.")

result_dir = os.path.join(work_dir, "RESULTS")
os.makedirs(result_dir, exist_ok=True)

print(f"📂 Répertoire : {repertoire}")
print(f"📄 Fichier : {fichier}")
print(f"💾 Résultats : {result_dir}")

### 📋 Afficher les métadonnées générales du fichier NetCDF

In [ ]:
!gdalinfo {os.path.join(repertoire, fichier)} 2>/dev/null 

### 🎯 Détail d’un sous‑dataset : le canal visible 0.6 µm (`/data/vis_06/measured/effective_radiance`)

In [ ]:
subdataset_path = "/data/vis_06/measured/effective_radiance"
netcdf_str = f'NETCDF:"{os.path.join(repertoire, fichier)}":{subdataset_path}'
!gdalinfo {netcdf_str} 2>/dev/null

<div class="alert alert-info alert-success">
    
## 4 - Extraction d’un canal et création d’une image
</div>

### 🖼️ Convertir le canal brut en GeoTIFF 8 bits (étirement linéaire 0–10000 → 0–255)

In [ ]:
out_tif_base = os.path.join(result_dir, "chunk_brute.tif")
!gdal_translate -scale 0 10000 0 255 -ot byte {netcdf_str} {out_tif_base} 2>/dev/null
print(f"✅ Image brute générée : {out_tif_base}")
!magick {out_tif_base} RESULTS/vis06_brute.jpg
im = Image.open(result_dir + '/vis06_brute.jpg', 'r')
display(im)

### 🎚️ Correction gamma (éclaircissement)

L’image brute est trop sombre. On applique un gamma = 2 (exposant = 1/2 = 0.5).

In [ ]:
out_tif_gamma = os.path.join(result_dir, "vis06_brute_gamma.tif")
!gdal_translate {out_tif_base} {out_tif_gamma} -scale -exponent 0.5 2>/dev/null
!magick -resize  {out_tif_gamma} RESULTS/vis06_brute_gamma.jpg
im2 = Image.open(result_dir + '/vis06_brute_gamma.jpg', 'r')
display(im2)
print(f"✅ Gamma appliqué : {out_tif_gamma}")

<div class="alert alert-info alert-success">
    
## 5 - Téléchargement et assemblage de plusieurs chunks
</div>

### ⬇️ Télécharger tous les chunks de 20 à 29 (même slot horaire)

In [ ]:
!eumdac download -c EO:EUM:DAT:0662 \
    --start 2025-05-14T10:00:00 --end 2025-05-14T10:10:00 \
    --entry '*002?.nc' \
    --output {output_chunk_dir} 2>/dev/null
print("✅ Chunks 20-29 téléchargés.")

### 🔄 Conversion de chaque chunk en GeoTIFF (étirement + gamma 0.3)

In [ ]:
# Déterminer le répertoire contenant ces chunks (normalement le même que précédemment)
chunk_dir_multi = glob.glob(os.path.join(output_chunk_dir, "W_XX-EUMETSAT-Darmstadt*"))[0]
subdataset_vis = "/data/vis_06/measured/effective_radiance"

for nc_file in glob.glob(os.path.join(chunk_dir_multi, "*.nc")):
    # Filtrer sur les chunks 20-29 (le nom contient *002?.nc)
    if not "002" in os.path.basename(nc_file):
        continue
    sub = f'NETCDF:"{nc_file}":{subdataset_vis}'
    out_tif = os.path.join(result_dir, f"{os.path.basename(nc_file)[:-3]}_scaled.tif")
    subprocess.run([
        "gdal_translate", sub, out_tif,
        "-scale", "0", "10000", "0", "255",
        "-exponent", "0.3",
        "-ot", "Byte"
    ], stderr=subprocess.DEVNULL)
    print(f"✅ {out_tif}")

### 🧩 Assemblage des chunks en une seule image (mosaïque)

In [ ]:
mosaic_tif = os.path.join(result_dir, "image_chunks20a29.tif")
!gdalwarp -q {result_dir}/*002?_scaled.tif {mosaic_tif}
print(f"✅ Mosaïque générée : {mosaic_tif}")

In [ ]:
mosaic_jpg = os.path.join(result_dir, "image_chunks20a29.jpg")
!magick convert -resize 2000 {mosaic_tif} {mosaic_jpg}
display(Image.open(mosaic_jpg))

<div class="alert alert-info alert-success">
    
## 6 - Visualisation d’un plein disque
</div>

### 🌍 Données pré-téléchargées d’un slot complet (tous les chunks)

Ces données sont disponibles dans `/stockage/DATA/202505141000`. Vous pouvez également les télécharger vous‑même avec :
```bash
eumdac download -c EO:EUM:DAT:0662 --start 2025-05-14T10:00:00 --end 2025-05-14T10:10:00 --output /stockage/DATA/202505141000
```

In [ ]:
nc_folder = "/stockage/DATA/202505141000"
full_disk_dir = os.path.join(result_dir, "plein_disque")
os.makedirs(full_disk_dir, exist_ok=True)

for nc_file in glob.glob(os.path.join(nc_folder, "*.nc")):
    sub = f'NETCDF:"{nc_file}":{subdataset_vis}'
    out_tif = os.path.join(full_disk_dir, f"{os.path.basename(nc_file)[:-3]}_scaled.tif")
    subprocess.run([
        "gdal_translate", sub, out_tif,
        "-scale", "0", "10000", "0", "255",
        "-exponent", "0.3",
        "-ot", "Byte"
    ], stderr=subprocess.DEVNULL)
    print(f"✅ {out_tif}")

# Mosaïque complète
full_mosaic = os.path.join(full_disk_dir, "image_plein_disque.tif")
!gdalwarp {full_disk_dir}/*_scaled.tif {full_mosaic}
print(f"✅ Mosaïque plein disque : {full_mosaic}")

In [ ]:
full_jpg = os.path.join(result_dir, "image_plein_disque.jpg")
!magick convert -resize 1000x1000 {full_mosaic} {full_jpg}
display(Image.open(full_jpg))

<div class="alert alert-info alert-success">
    
## 7 - Téléchargement direct d’une sous-zone (Data Tailor)
</div>

### ✂️ Utiliser le service *Data Tailor* pour obtenir un GeoTIFF prédécoupé

Cette approche permet de récupérer directement un produit géoréférencé sur une région d’intérêt (ROI).  
**Exemple** : canal IR_038 (haute résolution) sur l’Afrique de l’Ouest, format GeoTIFF.

In [ ]:
tailor_dir = os.path.join(result_dir, "tailor")
os.makedirs(tailor_dir, exist_ok=True)

!eumdac download -c EO:EUM:DAT:0665 \
    --start 2025-09-29T07:00:00 --end 2025-09-29T07:10:00 \
    --tailor "format: geotiff, product: FCIL1HRFI, roi: west_africa, projection: geographic, filter: {{bands: [ir_38_hr_effective_radiance]}}" \
    --output {tailor_dir} 2>/dev/null
print("✅ Téléchargement tailor effectué.")

### 🖼️ Affichage du résultat

In [ ]:
# Rechercher le fichier GeoTIFF produit
tif_files = glob.glob(os.path.join(tailor_dir, "*.tif"))
if tif_files:
    tailor_tif = tif_files[0]
    tailor_jpg = os.path.join(tailor_dir, "fichier.jpg")
    !convert -resize 1000x1000 {tailor_tif} {tailor_jpg}
    display(Image.open(tailor_jpg))
else:
    print("⚠️ Aucun fichier GeoTIFF trouvé. Vérifiez la requête.")

---

## ✅ Récapitulatif des commandes essentielles

| Action | Commande |
|--------|----------|
| Rechercher des produits | `eumdac search -c COLLECTION --limit N` |
| Télécharger un produit complet | `eumdac download -c COLLECTION --limit 1` |
| Télécharger des chunks spécifiques | `eumdac download -c COLLECTION --entry '*0036.nc'` |
| Appliquer un tailoring (format, ROI, bandes) | `eumdac download ... --tailor "..."` |
| Lire un sous‑dataset NetCDF avec GDAL | `gdalinfo NETCDF:"fichier.nc":canal` |
| Extraire un canal en JPEG/GeoTIFF | `gdal_translate -scale ...` |
| Assembler plusieurs chunks | `gdalwarp fichier1.tif fichier2.tif ... mosaic.tif` |

---

## 🔗 Ressources utiles

- [Guide EUMDAC](https://user.eumetsat.int/resources/user-guides/eumetsat-data-access-client-eumdac-guide)
- [Catalogue Data Store](https://data.eumetsat.int/search)
- [Documentation GDAL](https://gdal.org)

---

✅ **Fin du TP**